# Microdati → DataFrame

Questo notebook:
1. Legge il **tracciato** (file HTML) per ricavare nome, posizione, lunghezza e tipo di ogni campo
2. Segue i link nel tracciato per costruire le **tabelle di decodifica**
3. Legge i **file di microdati** (fixed-width) per tutti gli anni
4. Applica le decodifiche ai campi categoriali
5. Esporta il DataFrame finale

**Prima di eseguire:** configura le variabili nella cella `# CONFIGURAZIONE`.

In [ ]:
# Installa le dipendenze necessarie
!pip install -q beautifulsoup4 lxml pandas

In [ ]:
# Monta Google Drive (se i file sono su Drive)
# Commenta questo blocco se lavori in locale o su Colab con upload diretto
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURAZIONE — modifica questi parametri
# ============================================================
import os

# Cartella base in cui si trovano tutti i dati
# Esempio Drive:  '/content/drive/MyDrive/microdati'
# Esempio locale: '/content/microdati'
BASE_DIR = '/content/drive/MyDrive/microdati'

# Path del file HTML del tracciato
# Può essere nella stessa cartella dei dati o in una posizione a parte
TRACCIATO_HTML = os.path.join(BASE_DIR, 'tracciato.html')

# Cartella in cui si trovano gli HTML delle tabelle di decodifica
# (di solito la stessa del tracciato; i link nell'HTML sono relativi a questa cartella)
DECODE_DIR = os.path.dirname(TRACCIATO_HTML)

# Pattern per trovare i file di microdati tra cartelle e anni.
# Usa glob per specificare il percorso. Esempi:
#   - tutti i .txt in qualsiasi sottocartella:  '**/*.txt'
#   - una cartella per anno:                    '*/dati.txt'
#   - file con anno nel nome:                   'microdati_*.txt'
DATA_GLOB_PATTERN = '**/*.txt'

# Encoding dei file di microdati (tipicamente 'latin-1' per dati ISTAT)
DATA_ENCODING = 'latin-1'

# Encoding degli HTML del tracciato
HTML_ENCODING = 'latin-1'

# Colonne della tabella HTML del tracciato.
# Mappa il significato logico al nome (o indice 0-based) della colonna.
# Se usi l'indice, scrivi un intero; se usi il nome esatto dell'intestazione, scrivi una stringa.
TRACCIATO_COL_NOME       = 0   # nome del campo / variabile
TRACCIATO_COL_INIZIO     = 1   # posizione di inizio (1-based)
TRACCIATO_COL_LUNGHEZZA  = 2   # lunghezza del campo
TRACCIATO_COL_TIPO       = 3   # tipo (N=numerico, C=carattere, ecc.)
TRACCIATO_COL_DESCR      = 4   # descrizione testuale (opzionale)

# Formato di esportazione: 'csv', 'parquet', 'pickle'
EXPORT_FORMAT = 'parquet'

# Path del file di output
OUTPUT_FILE = os.path.join(BASE_DIR, f'microdati_all.{EXPORT_FORMAT}')

# Se True, sostituisce i codici numerici con le etichette delle tabelle di decodifica
APPLY_DECODING = True

# Se True, aggiunge una colonna 'anno' ricavata dal path del file
ADD_YEAR_COLUMN = True

print('Configurazione caricata.')

In [ ]:
import re
import glob
import warnings
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')
print('Librerie importate.')

## 1. Parsing del tracciato HTML

In [ ]:
def _get_cell(cells, col_ref):
    """Restituisce il testo di una cella tramite indice intero o nome colonna."""
    if isinstance(col_ref, int):
        return cells[col_ref].get_text(strip=True) if col_ref < len(cells) else ''
    # se col_ref è una stringa, non usabile qui; restituisce stringa vuota
    return ''


def parse_tracciato(html_path, encoding=HTML_ENCODING):
    """
    Legge il file HTML del tracciato e restituisce:
      - fields: lista di dict con keys: nome, inizio (0-based), fine (0-based, escluso),
                lunghezza, tipo, descrizione
      - decode_links: dict {nome_campo: path_html_decodifica}
    """
    with open(html_path, encoding=encoding, errors='replace') as f:
        soup = BeautifulSoup(f, 'lxml')

    # Trova la prima tabella con abbastanza colonne
    table = None
    for t in soup.find_all('table'):
        rows = t.find_all('tr')
        if len(rows) > 1:
            max_cols = max(len(r.find_all(['td', 'th'])) for r in rows)
            if max_cols >= 3:
                table = t
                break

    if table is None:
        raise ValueError(f'Nessuna tabella trovata in {html_path}')

    rows = table.find_all('tr')

    # Rileva se la prima riga è intestazione
    first_row_cells = rows[0].find_all(['td', 'th'])
    has_header = any(c.name == 'th' for c in first_row_cells)
    start_row = 1 if has_header else 0

    # Se le colonne sono specificate per nome, costruisce mappatura
    col_map = {}
    if has_header and isinstance(TRACCIATO_COL_NOME, str):
        headers = [c.get_text(strip=True).lower() for c in first_row_cells]
        for logical, col_ref in [
            ('nome', TRACCIATO_COL_NOME),
            ('inizio', TRACCIATO_COL_INIZIO),
            ('lunghezza', TRACCIATO_COL_LUNGHEZZA),
            ('tipo', TRACCIATO_COL_TIPO),
            ('descrizione', TRACCIATO_COL_DESCR),
        ]:
            if isinstance(col_ref, str):
                col_map[logical] = headers.index(col_ref.lower()) if col_ref.lower() in headers else None
            else:
                col_map[logical] = col_ref
    else:
        col_map = {
            'nome': TRACCIATO_COL_NOME,
            'inizio': TRACCIATO_COL_INIZIO,
            'lunghezza': TRACCIATO_COL_LUNGHEZZA,
            'tipo': TRACCIATO_COL_TIPO,
            'descrizione': TRACCIATO_COL_DESCR,
        }

    fields = []
    decode_links = {}

    for row in rows[start_row:]:
        cells = row.find_all(['td', 'th'])
        if len(cells) < 3:
            continue

        nome = _get_cell(cells, col_map['nome'])
        inizio_raw = _get_cell(cells, col_map['inizio'])
        lunghezza_raw = _get_cell(cells, col_map['lunghezza'])
        tipo = _get_cell(cells, col_map['tipo']) if col_map.get('tipo') is not None else ''
        descrizione = _get_cell(cells, col_map['descrizione']) if col_map.get('descrizione') is not None else ''

        # Salta righe non valide (intestazioni annidate, righe vuote)
        if not nome or not inizio_raw.isdigit():
            continue

        inizio = int(inizio_raw) - 1  # converti a 0-based
        lunghezza = int(re.sub(r'\D', '', lunghezza_raw)) if lunghezza_raw else 0
        fine = inizio + lunghezza

        fields.append({
            'nome': nome,
            'inizio': inizio,
            'fine': fine,
            'lunghezza': lunghezza,
            'tipo': tipo.upper(),
            'descrizione': descrizione,
        })

        # Cerca link a tabella di decodifica nella riga
        for cell in cells:
            link = cell.find('a', href=True)
            if link:
                href = link['href']
                # Path assoluto rispetto alla cartella del tracciato
                decode_path = os.path.join(DECODE_DIR, href)
                decode_links[nome] = decode_path
                break

    print(f'Campi trovati: {len(fields)}')
    print(f'Campi con tabella di decodifica: {len(decode_links)}')
    return fields, decode_links


fields, decode_links = parse_tracciato(TRACCIATO_HTML)

# Anteprima
df_tracciato = pd.DataFrame(fields)
df_tracciato.head(10)

## 2. Parsing delle tabelle di decodifica

In [ ]:
def parse_decode_table(html_path, encoding=HTML_ENCODING):
    """
    Legge un HTML di decodifica e restituisce un dict {codice: etichetta}.
    Assume che ci sia almeno una tabella con due colonne: codice e descrizione.
    """
    if not os.path.exists(html_path):
        return {}

    with open(html_path, encoding=encoding, errors='replace') as f:
        soup = BeautifulSoup(f, 'lxml')

    mapping = {}
    for table in soup.find_all('table'):
        rows = table.find_all('tr')
        for row in rows:
            cells = row.find_all(['td', 'th'])
            if len(cells) >= 2:
                codice = cells[0].get_text(strip=True)
                etichetta = cells[1].get_text(strip=True)
                # Salta righe intestazione
                if codice and etichetta and not cells[0].name == 'th':
                    mapping[codice] = etichetta
        if mapping:
            break  # usa la prima tabella valida

    return mapping


# Carica tutte le tabelle di decodifica
decode_tables = {}
if APPLY_DECODING:
    for campo, html_path in decode_links.items():
        tbl = parse_decode_table(html_path)
        if tbl:
            decode_tables[campo] = tbl
            print(f'  {campo}: {len(tbl)} codici')
        else:
            print(f'  {campo}: [nessuna decodifica trovata in {html_path}]')

print(f'\nTabelle di decodifica caricate: {len(decode_tables)}')

## 3. Lettura dei file di microdati

In [ ]:
def extract_year_from_path(path):
    """Cerca un anno a 4 cifre (1900-2099) nel path del file."""
    m = re.search(r'((?:19|20)\d{2})', str(path))
    return int(m.group(1)) if m else None


def read_fixed_width_file(filepath, fields, encoding=DATA_ENCODING):
    """
    Legge un file di microdati a larghezza fissa usando le specifiche del tracciato.
    Restituisce un DataFrame.
    """
    colspecs = [(f['inizio'], f['fine']) for f in fields]
    names    = [f['nome'] for f in fields]

    df = pd.read_fwf(
        filepath,
        colspecs=colspecs,
        names=names,
        header=None,
        encoding=encoding,
        dtype=str,          # leggi tutto come stringa; converti dopo
    )
    return df


def convert_types(df, fields):
    """
    Converte i campi numerici (tipo 'N') in numeri.
    I campi non numerici restano stringhe (strip degli spazi).
    """
    for f in fields:
        col = f['nome']
        if col not in df.columns:
            continue
        df[col] = df[col].str.strip()
        if f['tipo'].startswith('N'):
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def apply_decodings(df, decode_tables):
    """
    Aggiunge colonne '<campo>_label' con l'etichetta decodificata.
    Lascia intatta la colonna originale con il codice.
    """
    for campo, mapping in decode_tables.items():
        if campo in df.columns:
            df[campo + '_label'] = df[campo].astype(str).str.strip().map(mapping)
    return df


# Trova tutti i file di dati
data_files = sorted(glob.glob(os.path.join(BASE_DIR, DATA_GLOB_PATTERN), recursive=True))
print(f'File trovati: {len(data_files)}')
for p in data_files:
    print(' ', p)

In [ ]:
# Leggi e processa tutti i file
all_dfs = []

for filepath in data_files:
    print(f'Lettura: {filepath} ...', end=' ')
    try:
        df = read_fixed_width_file(filepath, fields)
        df = convert_types(df, fields)

        if APPLY_DECODING:
            df = apply_decodings(df, decode_tables)

        if ADD_YEAR_COLUMN:
            year = extract_year_from_path(filepath)
            df.insert(0, 'anno', year)

        df['_source_file'] = os.path.basename(filepath)
        all_dfs.append(df)
        print(f'{len(df):,} righe, {len(df.columns)} colonne')
    except Exception as e:
        print(f'ERRORE: {e}')

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f'\nDataFrame totale: {len(df_all):,} righe x {len(df_all.columns)} colonne')
else:
    print('Nessun file letto con successo.')

## 4. Ispezione del DataFrame

In [ ]:
print(df_all.dtypes)
df_all.head(5)

In [ ]:
# Riepilogo per anno
if ADD_YEAR_COLUMN:
    display(df_all.groupby('anno').size().rename('righe').reset_index())

## 5. Esportazione

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

if EXPORT_FORMAT == 'csv':
    df_all.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
elif EXPORT_FORMAT == 'parquet':
    df_all.to_parquet(OUTPUT_FILE, index=False)
elif EXPORT_FORMAT == 'pickle':
    df_all.to_pickle(OUTPUT_FILE)
else:
    raise ValueError(f'Formato non supportato: {EXPORT_FORMAT}')

size_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
print(f'File salvato: {OUTPUT_FILE}  ({size_mb:.1f} MB)')

In [ ]:
# Opzionale: scarica il file direttamente nel browser
# (utile se non usi Google Drive)
# from google.colab import files
# files.download(OUTPUT_FILE)